# 05 — Extractive Question Answering

This notebook connects the BM25 retrieval results to a pretrained extractive QA model. It stores one fixed set of QA predictions per split so every abstention strategy can be compared using the same answer component.

## 1. Goal and model choice

We add an extractive question-answering component to the BM25 retriever. The proposal permits a pretrained QA model or a controlled extractive component; it does not require training a large language model.

We use `deepset/roberta-base-squad2`, a QA checkpoint trained for SQuAD 2.0-style answerable and unanswerable questions. It is larger than a tiny model, so start with a small sample and use CPU if GPU memory is limited.

In [ ]:
from pathlib import Path
import json, torch
import pandas as pd
from transformers import pipeline

ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
RESULTS=ROOT/"results"
RESULTS.mkdir(exist_ok=True)

MODEL_NAME="deepset/roberta-base-squad2"
device=0 if torch.cuda.is_available() else -1
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
qa_pipeline=pipeline(
    "question-answering",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device
)

## 2. Load question rows and retrieval feature files

We reconstruct top-k passages using the saved `retrieved_passage_ids` and the matching split corpus. Each question is evaluated against its retrieved contexts individually. We keep the QA model's score and no-answer probability where returned by the pipeline.

In [ ]:
def load_split(name):
    df=pd.read_csv(SPLITS/f"{name}.csv")
    df["answers"]=df["answers"].apply(json.loads)
    df["is_answerable"]=df["is_answerable"].astype(bool)
    return df

def load_retrieved(name):
    df=pd.read_csv(RESULTS/f"{name}_retrieval_features.csv")
    df["retrieved_passage_ids"]=df["retrieved_passage_ids"].apply(json.loads)
    return df

def load_corpus_for_split(name):
    questions=load_split(name)
    return questions[["paragraph_id","article_title","context"]].drop_duplicates("paragraph_id")

train_df=load_split("train")
val_df=load_split("validation")
test_df=load_split("test")
retrieved={"train":load_retrieved("train"),
           "validation":load_retrieved("validation"),
           "test":load_retrieved("test")}
corpora={name:load_corpus_for_split(name).set_index("paragraph_id") for name in retrieved}

## 3. Run QA on one example

The model extracts an answer span from a context. For SQuAD 2.0, it can also produce an empty answer when it judges the context unanswerable. The returned score is a model score, not a perfectly calibrated probability of correctness.

In [ ]:
example_q = train_df.iloc[0].question
example_context = train_df.iloc[0].context
example_result = qa_pipeline(
    question=example_q,
    context=example_context,
    handle_impossible_answer=True
)
example_result

## 4. QA over retrieved top-k passages

We ask the QA model about each retrieved passage separately, then select the candidate with the highest answer score. We also retain the best no-answer probability if the pipeline returns it. This simple aggregation is fixed for every decision strategy.

For speed, first test with `N_LIMIT=100`. Set it to `None` to process the complete split after confirming runtime and memory.

In [ ]:
N_LIMIT=100  # Change to None for the complete split after a successful small run.

def run_qa_for_split(split_name, top_k=5, limit=N_LIMIT):
    qdf=load_split(split_name)
    r_df=retrieved[split_name].set_index("question_id")
    corpus=corpora[split_name]
    if limit is not None:
        qdf=qdf.head(limit)

    outputs=[]
    for i,row in enumerate(qdf.itertuples(index=False), start=1):
        passage_ids=r_df.loc[row.question_id,"retrieved_passage_ids"]
        candidates=[]
        for pid in passage_ids[:top_k]:
            if pid not in corpus.index:
                continue
            context=corpus.loc[pid,"context"]
            pred=qa_pipeline(
                question=row.question,
                context=context,
                handle_impossible_answer=True
            )
            candidates.append({
                "passage_id":pid,
                "answer":pred.get("answer",""),
                "qa_score":float(pred.get("score",0.0)),
                "no_answer_probability":float(pred.get("no_answer_probability",0.0)),
                "context":context
            })

        if candidates:
            best=max(candidates,key=lambda x:x["qa_score"])
        else:
            best={"passage_id":None,"answer":"","qa_score":0.0,
                  "no_answer_probability":1.0,"context":""}

        outputs.append({
            "question_id":row.question_id,
            "predicted_answer":best["answer"],
            "qa_score":best["qa_score"],
            "qa_no_answer_probability":best["no_answer_probability"],
            "qa_passage_id":best["passage_id"],
        })
        if i % 25 == 0:
            print(f"{split_name}: processed {i}/{len(qdf)}")
    return pd.DataFrame(outputs)

qa_train=run_qa_for_split("train")
qa_val=run_qa_for_split("validation")
qa_test=run_qa_for_split("test")
display(qa_train.head())

## 5. Save predictions

For the first run, files may contain only the first 100 questions because `N_LIMIT=100`. After verifying the output, rerun with `N_LIMIT=None` to generate complete split predictions before continuing.

In [ ]:
for name, frame in [("train",qa_train),("validation",qa_val),("test",qa_test)]:
    frame.to_csv(RESULTS/f"{name}_qa_predictions.csv",index=False)
    print("Saved",name,len(frame))

## 6. Inspect QA behavior

Look at correct answers, incorrect answers and empty answers. This step is important: the abstention strategies will decide whether to trust or reject the model's attempt. Do not assume a high QA score guarantees correctness.

In [ ]:
def show_qa_examples(split_name, n=5):
    qdf=load_split(split_name)
    preds=pd.read_csv(RESULTS/f"{split_name}_qa_predictions.csv")
    merged=qdf.merge(preds,on="question_id")
    for r in merged.sample(min(n,len(merged)),random_state=42).itertuples():
        print("="*80)
        print("Question:",r.question)
        print("Gold answerable in paired passage:",r.is_answerable)
        print("Reference:",r.answer_text)
        print("Predicted:",r.predicted_answer)
        print("QA score:",r.qa_score)
        print("No-answer probability:",r.qa_no_answer_probability)

show_qa_examples("train")

## 7. Practical notes

- If the first run is too slow, keep a small sample while debugging.
- If GPU memory is insufficient, set `device=-1` for CPU inference.
- Do not compare strategies using different QA outputs.
- For final experiments, regenerate full predictions and preserve the model name and package versions.

**Next:** Notebook 06 implements always-answer and BM25-threshold abstention.